# Sessione 6 · Usare le API di IA

**Modulo 3 · Modelli e API** · 2 sessioni · LLM · prompt engineering · integrazione

> Oggi programmiamo sul serio con i modelli di linguaggio. Vedrai lo stesso esempio con **tre fornitori** per capire che il concetto è identico e cambiano solo i dettagli:
> 1. **Claude (Anthropic)** — a pagamento, con credito di prova gratuito alla registrazione
> 2. **OpenRouter** — nel cloud, con modelli **gratuiti** (Llama e altri)
> 3. **Ollama** — Llama eseguito **sul tuo stesso computer**, senza chiave né internet

**Al termine della sessione saprai:**
- Chiamare un'API di IA da Python (nelle sue tre varianti)
- Regolare i parametri chiave: temperatura, token e contesto
- Gestire gli errori con eleganza
- Risolvere tre casi reali: chatbot, riassunto e classificazione

*Risultato della sessione: chiami un'API di IA da Python e costruisci il tuo primo chatbot.*

## ⚠️ Prima di iniziare: una nota sincera su questo notebook

Le celle che chiamano le API **hanno bisogno delle tue chiavi personali** (o, per Ollama, del modello installato sulla tua macchina), quindi **non si eseguono da sole**: dovrai inserire la tua chiave quando richiesta. Il codice è scritto seguendo la documentazione ufficiale vigente di ogni fornitore (giugno 2026), ma ricorda che **questi servizi cambiano**: se il nome di un modello smette di esistere, cercalo sul sito del fornitore e sostituiscilo. Ti indico in ogni caso dove guardare.

**Regola d'oro di sicurezza:** non scrivere mai la tua chiave direttamente nel codice né caricarla su internet. Useremo `getpass` per inserirla senza che resti salvata nel notebook.

## 1. Installazione delle librerie

Esegui questa cella una volta. Installa i tre client. Su Google Colab funziona così com'è.

In [ ]:
# Il simbolo ! esegue un comando del terminale dal notebook
!pip install anthropic openai ollama -q
print("Librerie installate ✅")

## 2. Configurare le chiavi in modo sicuro

`getpass` chiede la chiave da tastiera senza mostrarla a schermo. Devi compilare solo quelle dei servizi che userai.

- **Anthropic** — ottieni la tua chiave su `console.anthropic.com` (inizia per `sk-ant-...`)
- **OpenRouter** — ottieni la tua chiave su `openrouter.ai/keys` (inizia per `sk-or-...`)
- **Ollama** — non serve chiave (gira sul tuo computer)

In [ ]:
import os
from getpass import getpass

# Inserisci solo quelle che userai; premi Invio per saltarne una
os.environ["ANTHROPIC_API_KEY"] = getpass("Chiave Anthropic (Invio per saltare): ")
os.environ["OPENROUTER_API_KEY"] = getpass("Chiave OpenRouter (Invio per saltare): ")
print("Chiavi configurate (quelle che hai inserito).")

## 3. La stessa chiamata con i tre fornitori

Facciamo la domanda più semplice possibile a ciascuno. Osserva che **la struttura è sempre la stessa**: un messaggio dell'utente, il modello risponde, leggiamo il testo.

### Variante A · Claude (Anthropic)

In [ ]:
import anthropic

cliente_claude = anthropic.Anthropic()   # legge ANTHROPIC_API_KEY dall'ambiente

respuesta = cliente_claude.messages.create(
    model="claude-haiku-4-5",            # modello veloce ed economico (puoi usare claude-sonnet-4-6)
    max_tokens=300,
    messages=[
        {"role": "user", "content": "Spiegami cos'è un'API come se avessi 12 anni."}
    ],
)

print(respuesta.content[0].text)

### Variante B · OpenRouter (modello gratuito nel cloud)

OpenRouter offre un'API **compatibile con quella di OpenAI**, quindi usiamo la libreria `openai` cambiando l'indirizzo base. I modelli gratuiti terminano in `:free`.

> ⚠️ La lista dei modelli gratuiti cambia spesso. Se quello sotto desse errore, vai su `openrouter.ai/models`, filtra per *free* e copiane uno che termina in `:free`. I modelli gratuiti hanno limiti d'uso (richieste al minuto e al giorno).

In [ ]:
from openai import OpenAI

cliente_openrouter = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

respuesta = cliente_openrouter.chat.completions.create(
    model="meta-llama/llama-4-maverick:free",   # modello Llama gratuito
    messages=[
        {"role": "user", "content": "Spiegami cos'è un'API come se avessi 12 anni."}
    ],
)

print(respuesta.choices[0].message.content)

### Variante C · Ollama (Llama 100% locale)

Se hai installato Ollama (`ollama.com`) e scaricato un modello con `ollama pull llama3.2`, puoi eseguire Llama **sul tuo stesso computer, senza internet né chiave**. Ideale per dati sensibili.

> Richiede che il servizio Ollama sia avviato. Modelli leggeri: `llama3.2`. Più grande e capace: `llama3.3` (richiede più memoria).

In [ ]:
from ollama import chat

respuesta = chat(
    model="llama3.2",   # devi averlo scaricato prima con: ollama pull llama3.2
    messages=[
        {"role": "user", "content": "Spiegami cos'è un'API come se avessi 12 anni."}
    ],
)

print(respuesta.message.content)

### 💡 La cosa importante dei tre blocchi

Nota lo schema comune: **un client, una lista di messaggi con `role` e `content`, e leggere il testo della risposta**. L'unica cosa che cambia tra i fornitori è il nome del metodo e dove si trova il testo di output:

| | Creare il client | Chiamare | Leggere il testo |
|---|---|---|---|
| Claude | `anthropic.Anthropic()` | `cliente.messages.create(...)` | `r.content[0].text` |
| OpenRouter | `OpenAI(base_url=..., api_key=...)` | `cliente.chat.completions.create(...)` | `r.choices[0].message.content` |
| Ollama | `from ollama import chat` | `chat(...)` | `r.message.content` |

## 4. Un'unica funzione per non ripetere codice

Per il resto della sessione, definiamo **una sola funzione** `preguntar()` valida per i tre fornitori. Così ci concentriamo sui concetti e non sull'idraulica. Appoggiati a lei negli esercizi.

In [ ]:
def preguntar(prompt, proveedor="claude", system=None, temperatura=0.7, max_tokens=500):
    """Invia un prompt al fornitore scelto e restituisce il testo della risposta.

    proveedor: 'claude', 'openrouter' oppure 'ollama'
    system:    istruzioni di ruolo (opzionale)
    """
    if proveedor == "claude":
        cliente = anthropic.Anthropic()
        kwargs = {"model": "claude-haiku-4-5", "max_tokens": max_tokens,
                  "temperature": temperatura,
                  "messages": [{"role": "user", "content": prompt}]}
        if system:
            kwargs["system"] = system
        return cliente.messages.create(**kwargs).content[0].text

    elif proveedor == "openrouter":
        cliente = OpenAI(base_url="https://openrouter.ai/api/v1",
                         api_key=os.environ.get("OPENROUTER_API_KEY"))
        mensajes = []
        if system:
            mensajes.append({"role": "system", "content": system})
        mensajes.append({"role": "user", "content": prompt})
        r = cliente.chat.completions.create(model="meta-llama/llama-4-maverick:free",
                                            messages=mensajes, temperature=temperatura,
                                            max_tokens=max_tokens)
        return r.choices[0].message.content

    elif proveedor == "ollama":
        mensajes = []
        if system:
            mensajes.append({"role": "system", "content": system})
        mensajes.append({"role": "user", "content": prompt})
        return chat(model="llama3.2", messages=mensajes).message.content

    else:
        raise ValueError("proveedor deve essere 'claude', 'openrouter' oppure 'ollama'")

# Prova (cambia il fornitore in base alla chiave che hai)
print(preguntar("Dammi 3 idee di business sostenibile in una frase ciascuna.", proveedor="claude"))

## 5. I parametri che contano davvero

**Temperatura** — controlla la creatività. Va da 0 a 1 (circa):
- Vicino a **0** → risposte più deterministiche e concentrate. Usala per classificare, estrarre dati o quando vuoi precisione
- Vicino a **0.8-1** → risposte più varie e creative. Usala per brainstorming o scrittura

**max_tokens** — la lunghezza massima della risposta. Ricorda: un token ≈ 3 o 4 lettere. Limitarlo controlla il costo ed evita risposte infinite.

**Contesto / cronologia** — ciò che il modello "vede". Se vuoi che ricordi la conversazione, devi **accumulare i messaggi** e inviarli tutti. Lo vediamo nel chatbot.

In [ ]:
# Stesso prompt, due temperature. Eseguilo un paio di volte e confronta.
creativo = preguntar("Inventa un nome per un'app di orti urbani.",
                     proveedor="claude", temperatura=1.0)
preciso  = preguntar("Inventa un nome per un'app di orti urbani.",
                     proveedor="claude", temperatura=0.0)
print("Temperatura alta :", creativo)
print("Temperatura bassa:", preciso)

## 6. Gestire gli errori con eleganza

Le chiamate alle API **a volte falliscono**: cade la rete, si esaurisce il limite gratuito (errore 429), la chiave è sbagliata... Un programma serio non si rompe: **cattura l'errore e reagisce**. Per questo usiamo `try` / `except`.

In [ ]:
def preguntar_seguro(prompt, proveedor="claude"):
    try:
        return preguntar(prompt, proveedor=proveedor)
    except Exception as error:
        # In un caso reale potresti riprovare, aspettare, o provare un altro fornitore
        return f"⚠️ Impossibile completare la chiamata ({type(error).__name__}): {error}"

print(preguntar_seguro("Ciao, funzioni?", proveedor="claude"))

## 7. Caso 1 · Chatbot con memoria

Un chatbot deve **ricordare** ciò che è stato detto. Il trucco, come anticipato: salvare tutti i messaggi in una lista e **continuare ad aggiungerli**. A ogni turno inviamo la cronologia completa.

Qui lo facciamo con Claude. Per OpenRouter/Ollama l'idea è identica (lista di messaggi con `role` e `content`).

In [ ]:
historial = [
    {"role": "user", "content": "Ciao, mi chiamo Marta e sto avviando un vivaio di piante."},
]

cliente = anthropic.Anthropic()
r1 = cliente.messages.create(model="claude-haiku-4-5", max_tokens=200, messages=historial)
respuesta1 = r1.content[0].text
print("🤖", respuesta1)

# Salviamo la risposta dell'assistente e aggiungiamo un nuovo turno dell'utente
historial.append({"role": "assistant", "content": respuesta1})
historial.append({"role": "user", "content": "Ricordi come mi chiamo e di cosa mi occupo?"})

r2 = cliente.messages.create(model="claude-haiku-4-5", max_tokens=200, messages=historial)
print("🤖", r2.content[0].text)   # dovrebbe ricordare Marta e il suo vivaio

## 8. Caso 2 · Riassunto automatico

Un uso classico: incollare un testo lungo e chiedere un riassunto. Qui brilla un **system prompt** che fissi lo stile del riassunto.

In [ ]:
texto = """La trasformazione digitale non consiste solo nel comprare tecnologia, ma nel
ripensare processi, cultura e modello di business. Molte PMI iniziano digitalizzando compiti
amministrativi e, poco a poco, incorporano dati per prendere decisioni migliori. L'errore
abituale è concentrarsi sullo strumento e dimenticare le persone che devono usarlo."""

resumen = preguntar(
    prompt=f"Riassumi il seguente testo in un'unica frase chiara:\n\n{texto}",
    proveedor="claude",
    system="Sei un esperto nel sintetizzare idee. Rispondi in italiano, breve e diretto.",
    temperatura=0.2,   # bassa: vogliamo fedeltà, non creatività
)
print(resumen)

## 9. Caso 3 · Classificazione con few-shot

Recuperiamo il *few-shot* della Sessione 5, ora in codice. Chiediamo che restituisca **solo la categoria** per poter usare il risultato in un programma.

In [ ]:
def clasificar_sentimiento(comentario, proveedor="claude"):
    prompt = f"""Classifica il sentiment come POSITIVO, NEGATIVO o NEUTRO.
Rispondi SOLO con una di quelle tre parole, senza spiegazioni.

Commento: "Il prodotto è arrivato in ritardo e mal imballato"  -> NEGATIVO
Commento: "Funziona perfettamente, molto soddisfatto"           -> POSITIVO
Commento: "È quello che mi aspettavo, niente di più"             -> NEUTRO
Commento: "{comentario}" ->"""
    return preguntar(prompt, proveedor=proveedor, temperatura=0.0).strip()

# Proviamo con vari commenti
for c in ["L'assistenza è stata eccellente", "Ci ha messo un mese ad arrivare", "Svolge la sua funzione"]:
    print(f"{clasificar_sentimiento(c):10s} <- {c}")

### 🟢 Esercizio 1 · Cambia fornitore

Prendi uno qualsiasi dei tre casi (chatbot, riassunto o classificazione) ed eseguilo con `proveedor="openrouter"` o `proveedor="ollama"`. Cambia molto la qualità della risposta? Annota le differenze che osservi.

In [ ]:
# TODO: prova un caso con un altro fornitore
# Esempio:
# print(clasificar_sentimiento("Mi è piaciuto molto il servizio", proveedor="openrouter"))


### 🟢 Esercizio 2 · Il tuo classificatore

Adatta la funzione `clasificar_sentimiento` perché classifichi le email come `URGENTE` o `NON URGENTE` (riusa il few-shot che hai progettato nella Sessione 5). Provala con 3 email inventate.

In [ ]:
# TODO: crea la funzione clasificar_urgencia(correo, proveedor="claude")


**Soluzione di riferimento:**

In [ ]:
def clasificar_urgencia(correo, proveedor="claude"):
    prompt = f"""Classifica l'email come URGENTE o NON URGENTE.
Rispondi SOLO con una di quelle due opzioni.

Email: "Il server è fuori uso e non possiamo fatturare"       -> URGENTE
Email: "Potremmo vederci la settimana prossima?"                -> NON URGENTE
Email: "Promemoria: aggiorna il tuo profilo quando puoi"         -> NON URGENTE
Email: "{correo}" ->"""
    return preguntar(prompt, proveedor=proveedor, temperatura=0.0).strip()

for c in ["Il cliente minaccia di disdire oggi stesso",
          "Ti giro le foto dell'evento del mese scorso",
          "Il sito dei pagamenti dà errore a tutti gli utenti"]:
    print(f"{clasificar_urgencia(c):12s} <- {c}")

## 10. Riepilogo della sessione e chiusura del Modulo 3

- Chiamare un LLM segue sempre lo stesso schema: **client → lista di messaggi → leggere il testo della risposta**
- Hai tre vie equivalenti: **Claude** (qualità), **OpenRouter** (gratis nel cloud) e **Ollama** (locale e privato)
- I parametri chiave sono **temperatura** (creatività), **max_tokens** (lunghezza) e la **cronologia** (memoria)
- Programma sempre pensando agli **errori** (`try`/`except`); le API falliscono e i limiti gratuiti si esauriscono
- Con prompt ben progettati risolvi **chatbot, riassunto e classificazione** quasi senza codice

Ora sai integrare l'IA generativa nei tuoi programmi. **Questo è esattamente ciò che fa internamente un'app di IA reale.**

**Per il Modulo 4:** mettiamo tutto insieme in un progetto dall'inizio alla fine: dall'idea a un'app web pubblicata con Streamlit.

👉 **Porta già pensato un problema piccolo e concreto** che vorresti risolvere con un'app di IA. Sarà il tuo progetto.